<figure>
    <center> <img src="../../images/mai_banner.png"  style="width:1000px;height:180px;" ></center>
</figure>

# Laboratorio 08: Escalado de descriptores y coeficiente de aprendizaje

En este laboratorio experimentaremos con el escalado de descriptores (_feature scalin_) y el establecimiento del coeficiente de aprendizaje (_learning rate_).

## Dependencias
En el siguiente bloque importamos las dependencias necesarias para ejecutar nuestro laboratorio. El modificador `-q` significa modo silencioso

In [ ]:
%pip install -q numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import load_house_data, run_gradient_descent 
from lab_utils_multi import norm_plot, plt_equal_scale, plot_cost_i_w
from lab_utils_common import dlc
np.set_printoptions(precision=2)
plt.style.use('./deeplearning.mplstyle')

## Notación

| General<br />Notation | Descripción | Python (if applicable) |
|:---|:---|:---|
| $a$ | escalar, sin negrita |  |
| $\mathbf{a}$ | vector, en negrita |  |
| $\mathbf{A}$ | matriz, mayúscula en negrita |  |
| **Regresión** |  |  |
| $\mathbf{X}$ | matriz de ejemplos de entrenamiento | `X_train` |
| $\mathbf{y}$ | valores objetivo de los ejemplos de entrenamiento | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | $i$-ésimo ejemplo de entrenamiento | `X[i]`, `y[i]` |
| $m$ | número de ejemplos de entrenamiento | `m` |
| $n$ | número de características en cada ejemplo | `n` |
| $\mathbf{w}$ | parámetro: peso | `w` |
| $b$ | parámetro: sesgo (bias) | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | El resultado de evaluar el modelo en $\mathbf{x}^{(i)}$ parametrizado por $\mathbf{w},b$: $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$ | `f_wb` |
| $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$ | el gradiente o derivada parcial del costo con respecto al parámetro $w_j$ | `dj_dw[j]` |
| $\frac{\partial J(\mathbf{w},b)}{\partial b}$ | el gradiente o derivada parcial del costo con respecto al parámetro $b$ | `dj_db` |

# Planteamiento del problema

Al igual que en los laboratorios anteriores, usarás como ejemplo motivador la predicción del precio de viviendas. El conjunto de datos de entrenamiento contiene muchos ejemplos con 4 características (tamaño, habitaciones, pisos y antigüedad), mostrados en la tabla de abajo. Nota que en este laboratorio la característica Tamaño está en pies cuadrados (sqft). Este conjunto de datos es más grande que el del laboratorio anterior.

Queremos construir un modelo de regresión lineal con estos valores para poder predecir el precio de otras viviendas, por ejemplo, una casa de 1200 sqft, 3 habitaciones, 1 piso y 40 años de antigüedad.

## Conjunto de datos:
| Tamaño (sqft) | Número de habitaciones | Número de pisos | Antigüedad de la vivienda | Precio (miles de dólares) |
|:---|:---|:---|:---|:---|
| 952  | 2 | 1 | 65 | 271.5 |
| 1244 | 3 | 2 | 64 | 232   |
| 1947 | 3 | 2 | 17 | 509.8 |
| ...  | ... | ... | ... | ... |

In [ ]:
# Cargando el conjunto de datos
X_train, y_train = load_house_data()
X_features = ['size(sqft)','bedrooms','floors','age']

Vamos a generar una visualización del dataset y sus caracteristicas (_features_) graficando cada caracteristica vs el precio.

In [ ]:
# El argumento sharey=True asegura que todos los subgráficos compartan el mismo eje y
fig, ax=plt.subplots(1,4,figsize=(12,3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i], y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (Ks)")
# Mostrando el gráfico
plt.show()

Graficar cada característica contra el valor objetivo (el precio) da una idea de cuáles características influyen más en el precio. En las gráficas de arriba, al aumentar el tamaño también aumenta el precio. Las habitaciones y los pisos no parecen tener un impacto fuerte en el precio. Las casas más nuevas tienen precios más altos que las más antiguas.

## Descenso del gradiente con múltiples variables
Estas son las ecuaciones que desarrollaste en el laboratorio anterior sobre el descenso del gradiente para múltiples variables:

$$\begin{align*} \text{repetir}&\text{ hasta converger:} \; \lbrace \newline\;
& w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{para j = 0..n-1}\newline
&b\ \ := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

donde $n$ es el número de características, los parámetros $w_j$ y $b$ se actualizan simultáneamente, y donde

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3}
\end{align}
$$
* $m$ es el número de ejemplos de entrenamiento en el conjunto de datos

* $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ es la predicción del modelo, mientras que $y^{(i)}$ es el valor objetivo

## Coeficiente de aprendizaje
En la sesión técnica se discutieron algunos de los problemas relacionados con la elección del coeficiente de aprendizaje $\alpha$. El coeficiente de aprendizaje controla el tamaño de la actualización de los parámetros (ver la ecuación (1) arriba) y es compartida por todos los parámetros.

Ejecutemos el descenso del gradiente y probemos algunos valores de $\alpha$ en nuestro conjunto de datos.

### Coeficiente $\alpha$ = 9.9e-7

In [ ]:
# Estableciendo alpha en 9.9e-7
_, _, hist = run_gradient_descent(X_train, y_train, 10, alpha=9.9e-7)

El coeficiente de aprendizaje esta alto, esto se puede aprecioar en la columna costo. El costo se esta **incrementando** en cada iteración en lugar de decrecer. Vamos a graficar el resultado.

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

La gráfica de la derecha muestra el valor de uno de los parámetros, $w_0$. En cada iteración se pasa del valor óptimo (overshooting) y, como resultado, el costo termina *aumentando* en lugar de acercarse al mínimo. Ten en cuenta que esta no es una imagen completamente precisa, ya que en cada pasada se modifican 4 parámetros y no solo uno. Esta gráfica solo muestra $w_0$ con los demás parámetros fijos en valores benignos. En esta y en las siguientes gráficas puedes notar que las líneas azul y naranja están ligeramente desalineadas.

### Coeficiente $\alpha$ = 9e-7

Intentaremos con un valor mas pequeño.

In [ ]:
#Estableciendo un valor de alpha en 9e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 9e-7)

El costo esta decreciendo a lo largo de la ejecución monstrando que _alpha_ no es grande.

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

A la izquierda se ve que el costo disminuye, como debe ser. A la derecha se ve que $w_0$ sigue oscilando alrededor del mínimo, pero el costo disminuye en cada iteración en lugar de aumentar. Nota arriba que `dj_dw[0]` cambia de signo en cada iteración, ya que `w[0]` salta sobre el valor óptimo.
Este valor de alpha convergerá. Puedes variar el número de iteraciones para ver cómo se comporta.

### Coeficiente $\alpha$ = 1e-7

Intentaremos un valor aun mas pequeño para $\alpha$ y veamos que pasa.

In [ ]:
#set alpha to 1e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 1e-7)

El costo disminuye durante toda la ejecución, lo que muestra que $\alpha$ no es demasiado grande.

In [ ]:
plot_cost_i_w(X_train,y_train,hist)

A la izquierda se ve que el costo disminuye, como debe ser. A la derecha se ve que $w_0$ se acerca al mínimo sin oscilaciones. `dj_w0` es negativo durante toda la ejecución. Esta solución también convergerá.

## Escalado de características (Feature Scaling)
<figure>
    <center><img src="./img/feature_scaling.png" style="width:800px;" ></center>
</figure>

En las clases se describió la importancia de reescalar el conjunto de datos para que las características tengan un rango similar.
Si te interesan los detalles de por qué sucede esto, haz clic en el encabezado «details» de abajo. Si no, la siguiente sección explica paso a paso una implementación del escalado de características.

<details>
<summary>
    <font size='5', color='teal'><b>Detalles</b></font>
</summary>

Veamos de nuevo la situación con $\alpha$ = 9e-7. Este valor está bastante cerca del máximo que podemos asignar a $\alpha$ sin que el algoritmo diverja. Esta es una ejecución corta que muestra las primeras iteraciones:

<figure>
    <img src="./img/shortRun.png" style="width:1200px;" >
</figure>

Arriba se ve que, aunque el costo disminuye, $w_0$ avanza más rápido que los demás parámetros debido a que su gradiente es mucho mayor.

La gráfica de abajo muestra el resultado de una ejecución muy larga con $\alpha$ = 9e-7. Esto toma varias horas.

<figure>
    <img src="./img/longRun.png" style="width:1200px;" >
</figure>
    
Arriba se ve que el costo disminuyó lentamente después de su reducción inicial. Observa la diferencia entre `w0` y `w1`, `w2`, `w3`, así como entre `dj_dw0` y `dj_dw1-3`. `w0` alcanza su valor casi final muy rápido y `dj_dw0` disminuye rápidamente a un valor pequeño, lo que indica que `w0` está cerca de su valor final. Los otros parámetros se redujeron mucho más lentamente.

¿Por qué sucede esto? ¿Hay algo que podamos mejorar? Veamos:
<figure>
    <center> <img src="./img/scale.png"   ></center>
</figure>   

La figura de arriba muestra por qué los $w$ se actualizan de forma desigual:
- $\alpha$ es compartida por todas las actualizaciones de parámetros (los $w$ y $b$).
- el término de error común se multiplica por las características en el caso de los $w$ (no de $b$).
- las características varían significativamente en magnitud, lo que hace que algunas se actualicen mucho más rápido que otras. En este caso, $w_0$ se multiplica por 'Tamaño(sqft)', que generalmente es > 1000, mientras que $w_1$ se multiplica por 'numero de habitaciones', que generalmente es de 2 a 4.
    
La solución es el escalado de características (Feature Scaling).

En las clases se discutieron tres técnicas diferentes:
- Escalado de características (feature scaling): Es esencialmente, dividir cada característica positiva entre su valor máximo o, de forma más general, reescalar cada característica usando sus valores mínimo y máximo con $\frac{x-min}{max-min}$. Ambas formas normalizan las características al rango de -1 a 1; la primera funciona para características positivas, es simple y sirve bien para el ejemplo de la clase, y la segunda funciona para cualquier característica.
- Normalización por la media (_mean normalization_): $x_i := \dfrac{x_i - \mu_i}{max - min} $
- Normalización Z-score, que exploraremos a continuación.

### Normalización Z-score
Después de la normalización Z-score, todas las características tendrán una media de 0 y una desviación estándar de 1.

Para implementarla, ajusta tus valores de entrada con la siguiente fórmula:

$$x^{(i)}_j = \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{4}$$

donde $j$ selecciona una característica o una columna de la matriz $\mathbf{X}$, $\mu_j$ es la media de todos los valores de la característica $j$ y $\sigma_j$ es su desviación estándar:

$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=0}^{m-1} x^{(i)}_j \tag{5}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=0}^{m-1} (x^{(i)}_j - \mu_j)^2  \tag{6}
\end{align}
$$

> **Nota de implementación:** al normalizar las características, es importante
guardar los valores usados en la normalización: la media y la desviación estándar calculadas. Después de aprender los parámetros
del modelo, normalmente queremos predecir los precios de casas que no hemos
visto antes. Dado un nuevo valor x (área de la sala y número de
habitaciones), primero debemos normalizar x usando la media y la desviación estándar
que calculamos previamente con el conjunto de entrenamiento.

**Implementación**

In [ ]:
def zscore_normalize_features(X):
    """
    Calcula  X, zcore normalizada por columna
    
    Args:
      X (ndarray (m,n))     : datos de entrada, m ejemplos, n caracteristicas
      
    Returns:
      X_norm (ndarray (m,n)): input normalizado por columna
      mu (ndarray (n,))     : media de cada caracteristica
      sigma (ndarray (n,))  : desviacion estandar de cada caracteristica
    """
    # calcula la media de cada columna/característica
    mu     = np.mean(X, axis=0)                 # mu tendrá forma (n,)
    # calcula la desviación estándar de cada columna/característica
    sigma  = np.std(X, axis=0)                  # sigma tendrá forma (n,)
    # elemento a elemento: resta la media de esa columna a cada ejemplo y divide entre su desviación estándar
    X_norm = (X - mu) / sigma

    return (X_norm, mu, sigma)
 
# Podemos verificar nuestro trabajo con este codigo que hace lo mismo
# con scikit learn
#from sklearn.preprocessing import scale
#scale(X_orig, axis=0, with_mean=True, with_std=True, copy=True)

Veamos los pasos que intervienen en la normalización Z-score. La gráfica de abajo muestra la transformación paso a paso.

In [ ]:
mu     = np.mean(X_train,axis=0)   
sigma  = np.std(X_train,axis=0) 
X_mean = (X_train - mu)
X_norm = (X_train - mu)/sigma      

fig,ax=plt.subplots(1, 3, figsize=(12, 3))
ax[0].scatter(X_train[:,0], X_train[:,3])
ax[0].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[0].set_title("Sin normalizar")
ax[0].axis('equal')

ax[1].scatter(X_mean[:,0], X_mean[:,3])
ax[1].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[1].set_title(r"X - $\mu$")
ax[1].axis('equal')

ax[2].scatter(X_norm[:,0], X_norm[:,3])
ax[2].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[2].set_title(r"Z-score normalizada")
ax[2].axis('equal')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle("Distribucion de features antes, durante y despues de la normalizacion")
plt.show()

La gráfica de arriba muestra la relación entre dos de los parámetros del conjunto de entrenamiento, "age" y "size(sqft)". *Se grafican con la misma escala*.
- Izquierda: sin normalizar. El rango de valores o la varianza de la característica 'size(sqft)' es mucho mayor que la de age.
- Centro: el primer paso resta la media o promedio a cada característica, dejándolas centradas alrededor de cero. Es difícil ver la diferencia en la característica 'age', pero 'size(sqft)' claramente queda alrededor de cero.
- Derecha: el segundo paso divide entre la desviación estándar, dejando ambas características centradas en cero y con una escala similar.

Normalicemos los datos y comparémoslos con los datos originales.

In [ ]:
# normalize the original features
X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)
print(f"X_mu = {X_mu}, \nX_sigma = {X_sigma}")
print(f"Rango Pico a Pico por columna en datos originales (raw) X:{np.ptp(X_train,axis=0)}")   
print(f"Rango Pico a Pico por columna en Normalizada X:{np.ptp(X_norm,axis=0)}")

El rango pico a pico de cada columna se reduce, por efecto de la normalización, de un factor de miles a un factor de 2-3.

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3))
for i in range(len(ax)):
    norm_plot(ax[i],X_train[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count");
fig.suptitle("Distribucion de features antes de la normalizacion")
plt.show()
fig,ax=plt.subplots(1,4,figsize=(12,3))
for i in range(len(ax)):
    norm_plot(ax[i],X_norm[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count"); 
fig.suptitle("Distribucion de features despues de la normalizacion")

plt.show()

Observa que, arriba, el rango de los datos normalizados (eje x) está centrado alrededor de cero y es aproximadamente de +/- 2. Lo más importante es que el rango es similar para cada característica.

Volvamos a ejecutar nuestro algoritmo de descenso del gradiente con los datos normalizados.
Observa el valor **muchísimo mayor de alpha**. Esto acelerará el descenso del gradiente.

In [ ]:
w_norm, b_norm, hist = run_gradient_descent(X_norm, y_train, 1000, 1.0e-1, )

Las características escaladas logran resultados muy precisos **¡mucho, muchísimo más rápido!** Observa que el gradiente de cada parámetro es minúsculo al final de esta ejecución bastante corta. Una tasa de aprendizaje de 0.1 es un buen punto de partida para la regresión con características normalizadas.

Grafiquemos nuestras predicciones contra los valores objetivo. 

> Ten en cuenta que la predicción se hace con las características normalizadas, mientras que la gráfica se muestra con los valores originales de las características.

In [ ]:
#predict target using normalized features
m = X_norm.shape[0]
yp = np.zeros(m)
for i in range(m):
    yp[i] = np.dot(X_norm[i], w_norm) + b_norm

    # plot predictions and targets versus original features    
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color=dlc["dlorange"], label = 'predict')
ax[0].set_ylabel("Price"); ax[0].legend();
fig.suptitle("Objetivo vs predicción usando el modelo normalizado con z-score")
plt.show()

Los resultados se ven bien. Algunos puntos a notar:
- con múltiples características, ya no podemos tener una única gráfica que muestre los resultados contra las características.
- al generar la gráfica se usaron las características normalizadas. Cualquier predicción que use los parámetros aprendidos con un conjunto de entrenamiento normalizado también debe normalizarse.

**Predicción**
El objetivo de generar nuestro modelo es usarlo para predecir precios de viviendas que no están en el conjunto de datos. Predigamos el precio de una casa de 1200 sqft, 3 habitaciones, 1 piso y 40 años de antigüedad. Recuerda que debes normalizar los datos con la media y la desviación estándar obtenidas cuando se normalizaron los datos de entrenamiento.

In [ ]:
# First, normalize out example.
x_house = np.array([1200, 3, 1, 40])
x_house_norm = (x_house - X_mu) / X_sigma
print(x_house_norm)
x_house_predict = np.dot(x_house_norm, w_norm) + b_norm
print(f" Precio estimado de una casa con 1200 sqft, 3 dormitorios, 1 piso, 40 años = ${x_house_predict*1000:0.0f}")

**Contornos de costo**

Otra forma de ver el escalado de características es mediante los contornos de costo. Cuando las escalas de las características no coinciden, la gráfica de contorno del costo contra los parámetros es asimétrica.

En la gráfica de abajo, la escala de los parámetros está igualada. La gráfica de la izquierda es el contorno de costo de w[0] (los pies cuadrados) contra w[1] (el número de habitaciones) antes de normalizar las características. La gráfica es tan asimétrica que las curvas que completan los contornos no son visibles. En contraste, cuando las características están normalizadas, el contorno de costo es mucho más simétrico. El resultado es que las actualizaciones de los parámetros durante el descenso del gradiente pueden avanzar de forma equivalente para cada parámetro.

In [ ]:
plt_equal_scale(X_train, X_norm, y_train)

## ¡Muy bien 🦅!

En este laboratorio:
- utilizaste las rutinas de regresión lineal con múltiples características que desarrollaste en laboratorios anteriores
- exploraste el impacto de la tasa de aprendizaje $\alpha$ en la convergencia
- descubriste el valor del escalado de características con la normalización Z-score para acelerar la convergencia

## Referencias

Los datos de viviendas se obtuvieron del [conjunto de datos Ames Housing](http://jse.amstat.org/v19n3/decock.pdf).